<a href="https://colab.research.google.com/github/elsheppardo/hello-world/blob/main/Stage_3_BTC_ACB_Ledger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
STAGE 3: BTC ACB Ledger
==========================
Purpose: Build the running Adjusted Cost Base (ACB) for BTC across
         every buy and sell. This produces a running weighted-average
         cost per BTC that is used to calculate gain/loss on every
         disposition.

Output: Stage3_BTC_ACB_Ledger.xlsx

Method (CRA-compliant ACB):
  - Every BUY adds BTC to the pool at its CAD cost.
  - New ACB per BTC = (total CAD cost in pool) / (total BTC in pool)
  - Every SELL removes BTC from the pool at the current ACB per BTC.
  - ACB per BTC stays the SAME on a sell (weighted avg unchanged
    because both numerator and denominator shrink proportionally).
  - Gain/loss on a sell = (CAD proceeds) − (BTC sold × ACB per BTC)

Transactions tracked:
  2022-03-05   Paxos buy    2.93303373 BTC     $115,484 USD
  2022-03-10   Paxos buy    2.94941317 BTC     $117,027 USD
  2022-03-14   Paxos buy    1.41420188 BTC     $53,421 USD
  2022-03-xx   KuCoin sell  7.19297966 BTC    (to UST for Terra)
  2022-06-30   KuCoin buy   0.98106529 BTC     at $18,857/BTC
  2022-07-01   Paxos sell   0.98005529 BTC    (proceeds $19,931)
  2022-07-12   KuCoin buy   1.00000000 BTC     at $19,524/BTC
  2022-07-14   Paxos sell   0.99829120 BTC    (proceeds $20,653)
  2022-08-17   KuCoin buy   0.99027396 BTC     at $23,400/BTC
  2022-08-17   Paxos buy    1.00000000 BTC     at $23,430/BTC
  2023-01-14   Paxos sell   2.00000000 BTC    (proceeds $41,598)
  2023-03-22   Paxos buy    2.03796470 BTC     at $28,567/BTC
  2023-03-22   KuCoin sell  2.03796470 BTC     (proceeds $57,063)

How this sheet is used:
  - Stage 4 (2022 Events) reads col F (ACB per BTC) for BTC sell gain/loss.
  - Stage 5 (2023 Events) does the same for 2023 sells.
  - Final running BTC balance (col E, last row) should be ZERO — full exit.

How to verify:
  1. Buy rows show BTC going IN (positive col C) and CAD going IN (positive col D).
  2. Sell rows show BTC going OUT (negative col C) and CAD going OUT at ACB (negative col D).
  3. Col E (Running BTC) ends at 0.0 — confirms full exit.
  4. Col F (ACB per BTC): moves only on buys. Sells keep it flat.
  5. Final SUM of col D should match: total CAD still in pool = 0 (or very near zero).
"""

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FONT_NAME = "Arial"
styles = {
    'title':       Font(name=FONT_NAME, size=16, bold=True, color='1F4E79'),
    'section':     Font(name=FONT_NAME, size=12, bold=True, color='1F4E79'),
    'header':      Font(name=FONT_NAME, size=10, bold=True, color='FFFFFF'),
    'body':        Font(name=FONT_NAME, size=10),
    'body_bold':   Font(name=FONT_NAME, size=10, bold=True),
    'input':       Font(name=FONT_NAME, size=10, color='0000FF'),
    'formula':     Font(name=FONT_NAME, size=10, color='000000'),
    'link':        Font(name=FONT_NAME, size=10, color='008000'),
    'key_output':  Font(name=FONT_NAME, size=10, bold=True, color='006100'),
    'note':        Font(name=FONT_NAME, size=9,  italic=True, color='595959'),
    'header_fill': PatternFill('solid', fgColor='1F4E79'),
    'buy_fill':    PatternFill('solid', fgColor='E7F5E7'),    # Light green for buys
    'sell_fill':   PatternFill('solid', fgColor='FDE9E9'),    # Light red for sells
    'total_fill':  PatternFill('solid', fgColor='E2EFDA'),
    'verify_fill': PatternFill('solid', fgColor='FFF2CC'),
}
thin = Side(border_style='thin', color='BFBFBF')
cell_border = Border(left=thin, right=thin, top=thin, bottom=thin)

FMT_USD    = '"$"#,##0.00;("$"#,##0.00);"-"'
FMT_CAD    = '"$"#,##0.00;("$"#,##0.00);"-"'
FMT_BTC    = '#,##0.00000000'
FMT_PRICE  = '"$"#,##0.00'
FMT_FX     = '0.0000'


# ============================================================
# Transaction data
# ============================================================
# Each row: (date, event, side, btc_amount, usd_amount, usd_price, fx_rate, notes)
# side: 'buy' or 'sell'
# For buys: usd_amount is the cost (what was paid). For sells: usd_amount is proceeds.
# FX rates from Stage 1.

transactions = [
    # Feb 24 small KuCoin BTC deposit (source unknown; small amount)
    ("2022-02-24", "KuCoin BTC deposit (source unknown)",   "buy",  0.02147629,   824.00, 38360.00, 1.2734,
     "Small BTC deposit to KuCoin, source not identified. Treated as acquisition at Feb 24 price."),
    # 2022 BTC activity
    ("2022-03-05", "Paxos buy (25 fills aggregated)",       "buy",  2.93303373, 115484.03, 39376.29, 1.2621,
     "Buy with USD from Mar 4 wire deposit"),
    ("2022-03-10", "Paxos buy (15 fills aggregated)",       "buy",  2.94941317, 117026.78, 39679.14, 1.2740,
     "Buy with USD"),
    ("2022-03-14", "Paxos buy (14 fills aggregated)",       "buy",  1.41420188, 53420.78, 37776.45, 1.2810,
     "Buy with USD from Mar 9 wire"),
    # March BTC→UST aggregate: all BTC at KuCoin swapped.
    # Amount = 7.29293966 (Paxos withdrawals in Mar) + 0.02147629 (Feb 24 KC deposit) = 7.31441595 BTC
    # Proceeds = $293,496.54 UST sent to Terra
    # Implied avg fill: $293,496.54 / 7.31441595 = $40,125.03/BTC (within Mar 2022 range)
    ("2022-03-15", "KuCoin sell BTC → UST (aggregate)",     "sell", 7.31441595, 293496.54, 40125.03, 1.2745,
     "All KuCoin BTC swapped to UST across 7 separate transfers Mar 9-17. Proceeds = UST sent to Terra."),
    # Note: KuCoin "Trade History" CSV has each buy listed TWICE (duplicate rows).
    # Actual BTC bought on KuCoin = sum of the FIRST HALF of listed trades per day.
    # Confirmed via cross-check: KuCoin buy minus withdrawal fee = Paxos deposit amount.
    ("2022-06-30", "KuCoin buy BTC with USDT",              "buy",  0.98105529, 18499.97, 18857.23, 1.2886,
     "Rebuy 0.98 BTC on KuCoin to transfer back to Paxos"),
    ("2022-07-01", "Paxos sell BTC",                         "sell", 0.98005529, 19931.06, 20338.49, 1.2855,
     "Sell on Paxos (~$20k proceeds)"),
    ("2022-07-12", "KuCoin buy BTC with USDT",               "buy",  0.99879120, 19499.00, 19523.60, 1.3017,
     "Rebuy ~1 BTC on KuCoin"),
    ("2022-07-14", "Paxos sell BTC",                         "sell", 0.99829120, 20653.41, 20689.29, 1.3075,
     "Sell on Paxos"),
    ("2022-08-17", "KuCoin buy BTC with USDT",               "buy",  0.99996931, 23398.70, 23399.72, 1.2860,
     "Rebuy ~1 BTC on KuCoin"),
    ("2022-08-17", "Paxos buy BTC with USD",                 "buy",  1.00000000, 23430.39, 23430.39, 1.2860,
     "Buy on Paxos (using cash proceeds from earlier sells)"),
    # 2023 BTC activity
    ("2023-01-14", "Paxos sell 2 BTC",                       "sell", 2.00000000, 41598.12, 20807.30, 1.3389,
     "Large sell on Paxos (13 fills aggregated)"),
    ("2023-03-22", "Paxos buy 2.04 BTC",                     "buy",  2.03796470, 58217.23, 28566.53, 1.3699,
     "Rebuy on Paxos for final transfer to KuCoin then NDAX"),
    ("2023-03-22", "KuCoin sell 2.04 BTC (main)",           "sell", 2.03796470, 57063.00, 28000.00, 1.3699,
     "Main BTC disposition — sells just-bought Paxos BTC"),
    ("2023-03-22", "KuCoin buy 0.055 BTC (small rebuy)",    "buy",  0.05485424,  1513.90, 27600.00, 1.3699,
     "Small rebuy on KuCoin after main sell. This BTC was eventually disposed as part of late-2023 USDC withdrawals."),
    ("2023-11-30", "KuCoin small BTC dispose (estimated)",  "sell", 0.05485424,  2042.00, 37230.00, 1.3589,
     "Estimated disposal of residual 0.055 BTC before late-2023 tail withdrawals to NDAX. BTC/USD ~$37k in Nov 2023."),
]


def build_workbook(output_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "BTC_ACB_Ledger"

    widths = [12, 40, 8, 15, 14, 14, 14, 14, 14, 14, 40]
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w

    # Title
    ws.cell(row=1, column=1, value="BTC ACB Ledger — Running Weighted-Average Cost Basis").font = styles['title']
    ws.cell(row=2, column=1, value=(
        "Tracks every BTC buy/sell across Paxos and KuCoin. Col J (ACB per BTC) is the key output "
        "used by Stages 4 and 5 for gain/loss on BTC dispositions."
    )).font = styles['note']

    # Headers (row 4)
    headers = [
        "Date",               # A
        "Event",              # B
        "Side",               # C
        "BTC Amount",         # D  (positive for buys, for sells same sign, but col E applies sign)
        "BTC Added/(Disposed)",  # E  (signed: + for buy, - for sell)
        "USD Cost/Proceeds",  # F
        "FX Rate",            # G
        "CAD Cost/Proceeds",  # H
        "Running BTC",        # I
        "ACB per BTC (CAD)",  # J  ← KEY OUTPUT
        "Notes",              # K
    ]
    for i, h in enumerate(headers, 1):
        cell = ws.cell(row=4, column=i, value=h)
        cell.font = styles['header']
        cell.fill = styles['header_fill']
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = cell_border

    # Data rows
    start_row = 5
    for i, (date, event, side, btc, usd, price, fx, notes) in enumerate(transactions):
        r = start_row + i
        row_fill = styles['buy_fill'] if side == 'buy' else styles['sell_fill']

        # A: Date
        c = ws.cell(row=r, column=1, value=date); c.font = styles['body']
        # B: Event
        c = ws.cell(row=r, column=2, value=event); c.font = styles['body']
        # C: Side
        c = ws.cell(row=r, column=3, value=side.upper()); c.font = styles['body_bold']
        c.alignment = Alignment(horizontal='center')
        # D: BTC Amount (always positive — raw)
        c = ws.cell(row=r, column=4, value=btc); c.font = styles['input']; c.number_format = FMT_BTC
        # E: BTC signed for ledger purposes
        if side == 'buy':
            c = ws.cell(row=r, column=5, value=f"=D{r}")
        else:
            c = ws.cell(row=r, column=5, value=f"=-D{r}")
        c.font = styles['formula']; c.number_format = FMT_BTC
        # F: USD amount
        c = ws.cell(row=r, column=6, value=usd); c.font = styles['input']; c.number_format = FMT_USD
        # G: FX rate
        c = ws.cell(row=r, column=7, value=fx); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_FX
        # H: CAD value (signed — buys add to pool +, sells remove from pool using ACB)
        # For buys:  H = F * G   (CAD cost going into pool)
        # For sells: H = -(ACB per BTC before this sale) * D (BTC disposed)
        #           = -(col J of previous row) * col D of this row
        # We need a running calculation of ACB, so we reference previous row's J.
        if side == 'buy':
            c = ws.cell(row=r, column=8, value=f"=F{r}*G{r}")
        else:
            # ACB per BTC from previous row times BTC disposed (negative to remove from pool)
            c = ws.cell(row=r, column=8, value=f"=-J{r-1}*D{r}")
        c.font = styles['formula']; c.number_format = FMT_CAD

        # I: Running BTC
        if i == 0:
            # First row: running = signed E
            c = ws.cell(row=r, column=9, value=f"=E{r}")
        else:
            c = ws.cell(row=r, column=9, value=f"=I{r-1}+E{r}")
        c.font = styles['formula']; c.number_format = FMT_BTC

        # J: ACB per BTC (recalculates after each event)
        # = SUM of all CAD values in pool / running BTC
        # Using IFERROR because if running BTC is 0 we'd divide by zero
        c = ws.cell(row=r, column=10, value=f"=IFERROR(SUM($H${start_row}:H{r})/I{r},0)")
        c.font = styles['key_output']; c.number_format = FMT_CAD

        # K: Notes
        c = ws.cell(row=r, column=11, value=notes); c.font = styles['note']
        c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)

        # Apply row fill and border
        for col in range(1, 12):
            ws.cell(row=r, column=col).fill = row_fill
            ws.cell(row=r, column=col).border = cell_border

    # Summary block
    last_row = start_row + len(transactions) - 1
    summary_row = last_row + 2

    c = ws.cell(row=summary_row, column=1, value="Summary Checks"); c.font = styles['section']

    r = summary_row + 1
    ws.cell(row=r, column=1, value="Final BTC Balance (should be 0):").font = styles['body_bold']
    c = ws.cell(row=r, column=4, value=f"=I{last_row}"); c.font = styles['formula']; c.number_format = FMT_BTC
    c.fill = styles['total_fill']
    ws.cell(row=r, column=5, value="✓ confirms full exit from BTC").font = styles['note']

    r = summary_row + 2
    ws.cell(row=r, column=1, value="Total CAD cost put into BTC (all buys):").font = styles['body']
    c = ws.cell(row=r, column=4, value=f"=SUMIFS(H{start_row}:H{last_row},C{start_row}:C{last_row},\"BUY\")")
    c.font = styles['formula']; c.number_format = FMT_CAD

    r = summary_row + 3
    ws.cell(row=r, column=1, value="Total CAD value removed via ACB (all sells):").font = styles['body']
    c = ws.cell(row=r, column=4, value=f"=SUMIFS(H{start_row}:H{last_row},C{start_row}:C{last_row},\"SELL\")")
    c.font = styles['formula']; c.number_format = FMT_CAD

    r = summary_row + 4
    ws.cell(row=r, column=1, value="Sum of H column (should be ~0 since pool fully emptied):").font = styles['body_bold']
    c = ws.cell(row=r, column=4, value=f"=SUM(H{start_row}:H{last_row})")
    c.font = styles['formula']; c.number_format = FMT_CAD
    c.fill = styles['total_fill']

    # Legend
    r = summary_row + 7
    ws.cell(row=r, column=1, value="Legend").font = styles['section']
    r += 1
    c = ws.cell(row=r, column=1, value="  Green rows = BUY (adds BTC and CAD to pool)")
    c.font = styles['body']
    c.fill = styles['buy_fill']
    r += 1
    c = ws.cell(row=r, column=1, value="  Red rows = SELL (removes BTC and CAD at current ACB per BTC)")
    c.font = styles['body']
    c.fill = styles['sell_fill']
    r += 1
    ws.cell(row=r, column=1, value="  Yellow cells = FX rates to VERIFY against Bank of Canada").font = styles['note']
    r += 1
    ws.cell(row=r, column=1, value="  Green bold numbers = ACB per BTC (key output)").font = styles['note']

    # Notes
    r = summary_row + 13
    ws.cell(row=r, column=1, value="How ACB works (for reference)").font = styles['section']
    r += 1
    for line in [
        "• ACB per BTC (col J) = Running sum of CAD in pool ÷ Running BTC in pool.",
        "• On a BUY: both numerator and denominator grow → new weighted average.",
        "• On a SELL: both shrink proportionally → ACB per BTC unchanged.",
        "• Gain/loss on a SELL (computed in Stage 4/5) = CAD proceeds − (BTC sold × ACB per BTC at time of sale).",
        "• The 'ACB per BTC at time of sale' is col J of the sell row itself (which equals the previous row's J).",
    ]:
        ws.cell(row=r, column=1, value=line).font = styles['note']
        r += 1

    ws.freeze_panes = "A5"
    wb.save(output_path)
    print(f"Saved: {output_path}")
    print(f"Transactions tracked: {len(transactions)}")


if __name__ == "__main__":
    build_workbook("Stage3_BTC_ACB_Ledger.xlsx")